<a href="https://colab.research.google.com/github/ndhtfm/UTS_Wine-Quality-Prediction/blob/main/UTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍷 Wine Quality Classification

Notebook ini membangun model **klasifikasi** untuk memprediksi kualitas anggur berdasarkan fitur-fitur kimiawi dari setiap sampel.

Notebook dibagi menjadi **dua tahap yang independen**:

| Tahap | Isi | Kapan dijalankan |
|-------|-----|------------------|
| **Tahap 1 — Training** | Load data → preprocessing → training model → evaluasi → simpan model | Satu kali saat membangun model |
| **Tahap 2 — Deployment & Prediksi** | Load model tersimpan → preprocessing data baru → prediksi → simpan hasil | Setiap kali ada data testing baru |

> ⚠️ **Tahap 2 berdiri sendiri dan tidak bergantung pada variabel Tahap 1.**  
> Pastikan tiga file berikut sudah tersedia sebelum menjalankan Tahap 2:  
> `model_wine_quality.pkl` · `scaler_wine_quality.pkl` · `train_median.pkl`

---

### 📌 Informasi Dataset
| Keterangan | Detail |
|---|---|
| **Sumber data** | Dataset Wine Quality (Red Wine) |
| **Jumlah fitur** | 11 fitur kimiawi (keasaman, kadar alkohol, sulfat, dll.) |
| **Variabel target** | `quality` — skor kualitas anggur skala 0–10 (bilangan bulat) |
| **Tipe masalah** | Klasifikasi multi-kelas |
| **Catatan** | Nilai quality yang muncul di data umumnya 3–8; kelas ekstrem (0–2 dan 9–10) sangat jarang karena sangat sedikit sampel anggur dengan kualitas ekstrem di dunia nyata |

---

## ══════════════════════════════════════
## TAHAP 1 — TRAINING & SIMPAN MODEL
## ══════════════════════════════════════

### 1.1 Konfigurasi & Import Library

Sebelum memulai, tentukan nama kolom **target** dan kolom **ID** sesuai dataset yang digunakan.  
Dengan mendefinisikannya sebagai variabel di sini, seluruh notebook otomatis menyesuaikan tanpa perlu mengubah kode satu per satu.

In [ ]:
# ════════════════════════════════════════════════════
#   KONFIGURASI — sesuaikan dengan dataset Anda
# ════════════════════════════════════════════════════
TARGET_COL = 'quality'   # nama kolom target (variabel yang ingin diprediksi)
ID_COL     = 'Id'        # nama kolom ID (tidak digunakan sebagai fitur)
NIM_3_DIGIT = '078'      # ganti dengan 3 digit terakhir NIM Anda
# ════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Konfigurasi dan library berhasil dimuat!')
print(f'   Kolom target : {TARGET_COL}')
print(f'   Kolom ID     : {ID_COL}')

### 1.2 Load Data Training

Data training adalah dataset yang digunakan untuk **melatih model**. Dataset ini harus memiliki kolom fitur dan kolom target (`quality`) yang sudah diketahui nilainya.

In [ ]:
from google.colab import files
import io

print('📂 Silakan upload file data_training.csv ↓')
uploaded_train = files.upload()

In [ ]:
# Baca file yang diupload langsung dari memori (tidak bergantung pada nama file)
filename_train = list(uploaded_train.keys())[0]
df_train = pd.read_csv(io.BytesIO(uploaded_train[filename_train]))

print(f"✅ '{filename_train}' berhasil dibaca")
print(f'📌 Ukuran dataset  : {df_train.shape[0]} baris × {df_train.shape[1]} kolom')
print(f'📌 Daftar kolom    : {df_train.columns.tolist()}')
display(df_train.head())

### 1.3 Preprocessing Data

Preprocessing adalah tahap **persiapan data** sebelum dimasukkan ke model. Data mentah seringkali memiliki masalah seperti nilai hilang, duplikasi, atau skala fitur yang tidak seragam — semua ini perlu diselesaikan terlebih dahulu agar model dapat belajar dengan baik.

**Langkah-langkah yang dilakukan:**
1. **Cek & tangani missing values** — nilai kosong dapat mengganggu perhitungan model
2. **Cek & hapus duplikasi** — baris identik dapat membuat model bias
3. **Hapus kolom ID** — kolom ID hanya penanda baris, bukan informasi yang relevan untuk prediksi
4. **Pisahkan fitur dan target** — model belajar dari fitur (X) untuk memprediksi target (y)
5. **Train/validation split** — sebagian data disisihkan untuk menguji performa model
6. **Feature scaling** — menyeragamkan skala semua fitur agar tidak ada fitur yang mendominasi

In [ ]:
# ── 1. Missing Values ────────────────────────────────────────────────────────
# Missing value adalah sel yang kosong atau tidak memiliki nilai.
# Strategi: isi dengan MEDIAN kolom — lebih robust dibanding mean
# karena tidak terpengaruh oleh nilai ekstrem (outlier).
missing = df_train.isnull().sum()
print('🔍 Jumlah missing values per kolom:')
print(missing[missing > 0] if missing.sum() > 0 else '   ✅ Tidak ada missing values')

# Simpan median dari data training untuk digunakan kembali di Tahap 2.
# Tujuannya agar preprocessing data testing menggunakan statistik yang
# sama — bukan dihitung ulang dari data testing yang berbeda.
train_median = df_train.median(numeric_only=True)

if missing.sum() > 0:
    df_train.fillna(train_median, inplace=True)
    print('✅ Missing values diisi dengan median data training.')

# ── 2. Duplikasi ─────────────────────────────────────────────────────────────
# Baris duplikat dapat membuat model terlalu menghafal pola tertentu
# dan tidak belajar secara general.
dup = df_train.duplicated().sum()
print(f'\n🔍 Baris duplikat : {dup}')
if dup > 0:
    df_train.drop_duplicates(inplace=True)
    df_train.reset_index(drop=True, inplace=True)
    print(f'✅ Duplikasi dihapus. Ukuran baru: {df_train.shape}')
else:
    print('✅ Tidak ada duplikasi.')

# ── 3. Pisah Fitur dan Target ────────────────────────────────────────────────
# Kolom ID dihapus karena hanya penomoran administratif,
# bukan hasil pengukuran yang mempengaruhi kualitas anggur.
df_clean = df_train.drop(columns=[ID_COL])
X = df_clean.drop(columns=[TARGET_COL])   # fitur: semua kolom selain target
y = df_clean[TARGET_COL]                  # target: kolom yang ingin diprediksi

print(f'\n✅ Fitur (X) : {X.shape[1]} kolom, {X.shape[0]} baris')
print(f'✅ Target (y): {y.shape[0]} nilai')
print(f'📌 Nama fitur: {X.columns.tolist()}')

In [ ]:
# ── Visualisasi Distribusi Kelas Target ──────────────────────────────────────
# Penting untuk mengetahui apakah jumlah sampel antar kelas seimbang.
# Ketidakseimbangan (class imbalance) dapat membuat model cenderung
# memprediksi kelas mayoritas dan mengabaikan kelas minoritas.
print(f'🎯 Distribusi kelas {TARGET_COL}:')
print(y.value_counts().sort_index())

counts      = y.value_counts().sort_index()
all_classes = list(range(0, 11))   # skala quality 0–10
all_counts  = [counts.get(c, 0) for c in all_classes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart — menampilkan seluruh skala 0–10 agar konteks terlihat utuh
axes[0].bar(all_classes, all_counts, color=sns.color_palette('muted', 11))
axes[0].set_xlabel(f'Nilai {TARGET_COL}')
axes[0].set_ylabel('Jumlah Sampel')
axes[0].set_title(f'Distribusi Kelas {TARGET_COL} (skala 0–10)')
axes[0].set_xticks(all_classes)
for cls, cnt in zip(all_classes, all_counts):
    if cnt > 0:
        axes[0].text(cls, cnt + 1, str(cnt), ha='center', fontweight='bold')

# Pie chart — menampilkan proporsi hanya untuk kelas yang benar-benar ada
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(counts)))
axes[1].set_title(f'Proporsi Kelas {TARGET_COL}')

plt.suptitle(f'Distribusi Variabel Target: {TARGET_COL}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Interpretasi otomatis berdasarkan data
max_cls = counts.idxmax()
min_cls = counts.idxmin()
print(f'\n📌 Interpretasi:')
print(f'   Kelas terbanyak : quality {max_cls} ({counts[max_cls]} sampel)')
print(f'   Kelas tersedikit: quality {min_cls} ({counts[min_cls]} sampel)')
if counts.max() / counts.min() > 5:
    print('   ⚠️  Terdapat ketidakseimbangan kelas (class imbalance) yang signifikan.')
    print('       Akan ditangani dengan class_weight="balanced" pada model.')
else:
    print('   ✅ Distribusi kelas cukup seimbang.')

In [ ]:
# ── 4. Train / Validation Split ──────────────────────────────────────────────
# Data training dibagi menjadi dua bagian:
#   - 80% untuk melatih model
#   - 20% untuk mengevaluasi performa model (data validasi)
# Parameter stratify=y memastikan proporsi tiap kelas tetap terjaga
# di kedua bagian — penting terutama saat data tidak seimbang.
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,   # hasil dapat direproduksi
    stratify=y
)

print(f'✅ Data latih    : {X_train.shape[0]} sampel  (80%)')
print(f'✅ Data validasi : {X_val.shape[0]} sampel  (20%)')

# ── 5. Feature Scaling ───────────────────────────────────────────────────────
# Fitur-fitur dalam dataset seringkali memiliki satuan dan rentang nilai
# yang sangat berbeda. Misalnya, kadar alkohol (8–15) vs total sulfur
# dioxide (0–300). Tanpa scaling, model bisa bias ke fitur berskala besar.
#
# StandardScaler mengubah setiap fitur menjadi: z = (x - mean) / std
# Hasilnya: semua fitur memiliki mean ≈ 0 dan std ≈ 1.
#
# ATURAN PENTING:
#   fit_transform() → hanya pada data LATIH (mean & std dihitung dari sini)
#   transform()     → pada data lainnya (validasi, testing)
# Tujuan: mencegah "kebocoran" informasi dari data validasi/testing ke model.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

print('✅ Feature scaling selesai! Semua fitur kini ber-mean ≈ 0 dan std ≈ 1.')

### 1.4 Training Model

Model yang digunakan adalah **Random Forest Classifier** — sebuah metode *ensemble* yang membangun banyak *decision tree* secara paralel, kemudian menggabungkan hasil prediksi dari semua pohon melalui mekanisme **majority voting** (suara terbanyak menentukan hasil akhir).

**Alasan memilih Random Forest:**
- Tidak memerlukan asumsi distribusi data tertentu
- Robust terhadap outlier dan fitur yang tidak relevan
- Parameter `class_weight='balanced'` secara otomatis memberikan bobot lebih pada kelas minoritas untuk menangani *class imbalance*
- Menghasilkan informasi *feature importance* sebagai analisis tambahan

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,        # jumlah pohon keputusan yang dibangun
    max_depth=None,          # pohon tumbuh penuh hingga semua daun murni
    min_samples_split=5,     # minimal 5 sampel diperlukan untuk membelah sebuah node
    class_weight='balanced', # bobot kelas otomatis untuk menangani class imbalance
    random_state=42          # seed acak untuk hasil yang dapat direproduksi
)

model.fit(X_train_scaled, y_train)

print('✅ Model berhasil dilatih!')
print(f'   Jumlah pohon : {model.n_estimators}')
print(f'   Kelas        : {list(model.classes_)}')

### 1.5 Evaluasi Model

Evaluasi dilakukan pada **data validasi** — bagian data training yang sengaja disisihkan dan tidak pernah dilihat model selama proses training. Ini memberikan gambaran seberapa baik model bekerja pada data baru yang belum pernah dilihat sebelumnya.

Dua metrik yang digunakan:
- **Akurasi**: proporsi prediksi yang benar dari seluruh prediksi → `benar / total`
- **Confusion Matrix**: matriks yang menampilkan rincian benar/salah prediksi per kelas

In [ ]:
# Prediksi pada data latih dan validasi
y_pred_train = model.predict(X_train_scaled)
y_pred_val   = model.predict(X_val_scaled)

acc_train = accuracy_score(y_train, y_pred_train)
acc_val   = accuracy_score(y_val,   y_pred_val)

print('=' * 50)
print('           📊 HASIL EVALUASI MODEL')
print('=' * 50)
print(f'  Akurasi Data Latih    : {acc_train*100:.2f}%')
print(f'  Akurasi Data Validasi : {acc_val*100:.2f}%')
print('=' * 50)

# Interpretasi selisih akurasi latih vs validasi
gap = acc_train - acc_val
print(f'\n📌 Interpretasi:')
if gap > 0.15:
    print(f'   Selisih akurasi: {gap:.2f} → model kemungkinan overfitting.')
    print('   Overfitting terjadi ketika model terlalu menghafal data latih')
    print('   sehingga performa pada data baru menurun.')
else:
    print(f'   Selisih akurasi: {gap:.2f} → model generalisasi dengan baik.')
    print('   Model tidak hanya menghafal data latih, tetapi mampu')
    print('   memprediksi data baru dengan performa yang konsisten.')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
# Confusion Matrix adalah tabel n×n (n = jumlah kelas) yang menampilkan
# rincian prediksi model:
#   - Baris    = kelas AKTUAL (nilai sebenarnya)
#   - Kolom    = kelas PREDIKSI (nilai yang diprediksi model)
#   - Diagonal = prediksi BENAR (aktual = prediksi)
#   - Lainnya  = prediksi SALAH (aktual ≠ prediksi)
labels = sorted(y.unique())
cm = confusion_matrix(y_val, y_pred_val, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=labels, yticklabels=labels,
    cmap='Blues', linewidths=0.5
)
plt.xlabel('Prediksi', fontsize=11)
plt.ylabel('Aktual', fontsize=11)
plt.title(f'Confusion Matrix — Data Validasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Hitung akurasi per kelas dari confusion matrix
print('📌 Interpretasi Confusion Matrix:')
print('   - Sel diagonal (biru gelap) = jumlah prediksi BENAR per kelas')
print('   - Sel luar diagonal         = jumlah prediksi SALAH (tertukar ke kelas lain)')
print('   - Kelas dengan sampel sedikit cenderung memiliki akurasi lebih rendah')
print(f'\n   Total prediksi benar  : {cm.diagonal().sum()} dari {cm.sum()} sampel validasi')

### 1.6 Simpan Model & Scaler

Model dan scaler disimpan ke file `.pkl` menggunakan `joblib`. Penyimpanan ini penting agar:
- Model tidak perlu dilatih ulang setiap kali ingin melakukan prediksi
- Scaler yang sama digunakan pada data training dan testing — konsistensi preprocessing terjaga
- Model dapat didistribusikan dan digunakan di environment lain

In [ ]:
# Simpan model dan scaler ke file
# Keduanya harus disimpan bersama karena preprocessing pada data testing
# harus menggunakan mean & std yang SAMA dengan saat training.
joblib.dump(model,  'model_wine_quality.pkl')
joblib.dump(scaler, 'scaler_wine_quality.pkl')
joblib.dump(train_median, 'train_median.pkl')

print('File yang tersimpan:')
print('  💾 model_wine_quality.pkl   → model klasifikasi Random Forest')
print('  💾 scaler_wine_quality.pkl  → StandardScaler yang sudah dilatih')
print('  💾 train_median.pkl         → median data training (untuk imputasi)')
print(f'\n📊 Akurasi model pada data validasi : {acc_val*100:.2f}%')
print('\n✅ Tahap 1 selesai. Simpan ketiga file .pkl tersebut.')
print('   Lanjutkan ke Tahap 2 saat data testing sudah tersedia.')

---

## ══════════════════════════════════════
## TAHAP 2 — DEPLOYMENT & PREDIKSI
## ══════════════════════════════════════

Tahap ini adalah proses **deployment** — menggunakan model yang sudah dilatih untuk memprediksi data baru.

> 🔁 **Tahap ini berdiri sendiri dan tidak memerlukan Tahap 1 dijalankan ulang.**  
> Pastikan tiga file berikut sudah tersedia di direktori yang sama:
> - `model_wine_quality.pkl`
> - `scaler_wine_quality.pkl`  
> - `train_median.pkl`

### 2.1 Konfigurasi & Import Library

In [ ]:
# ════════════════════════════════════════════════════
#   KONFIGURASI — sesuaikan dengan dataset Anda
# ════════════════════════════════════════════════════
ID_COL      = 'Id'    # nama kolom ID pada data testing
NIM_3_DIGIT = '078'   # ganti dengan 3 digit terakhir NIM Anda
# ════════════════════════════════════════════════════

import pandas as pd
import joblib

print('✅ Library berhasil diimport!')

### 2.2 Load Model, Scaler & Median Training

Ketiga file ini dimuat dari hasil penyimpanan di Tahap 1. Menggunakan file yang sama memastikan proses prediksi konsisten dengan proses training.

In [ ]:
loaded_model  = joblib.load('model_wine_quality.pkl')
loaded_scaler = joblib.load('scaler_wine_quality.pkl')
train_median  = joblib.load('train_median.pkl')

print('✅ Model, scaler, dan median berhasil dimuat!')
print(f'   Tipe model    : {type(loaded_model).__name__}')
print(f'   Kelas target  : {list(loaded_model.classes_)}')
print(f'   Jumlah fitur  : {loaded_model.n_features_in_}')

### 2.3 Load Data Testing

Data testing adalah data baru **tanpa kolom target** — inilah yang akan diprediksi oleh model.

In [ ]:
from google.colab import files
import io

print('📂 Silakan upload file data_testing.csv ↓')
uploaded_test = files.upload()

In [ ]:
# Baca file yang diupload langsung dari memori
filename_test = list(uploaded_test.keys())[0]
df_test = pd.read_csv(io.BytesIO(uploaded_test[filename_test]))

print(f"✅ '{filename_test}' berhasil dibaca")
print(f'📌 Ukuran dataset  : {df_test.shape[0]} baris × {df_test.shape[1]} kolom')
print(f'📌 Daftar kolom    : {df_test.columns.tolist()}')
display(df_test.head())

### 2.4 Preprocessing Data Testing

Data testing harus melalui preprocessing yang **identik** dengan data training agar model dapat memprosesnya dengan benar. Yang membedakan hanyalah:
- Missing values diisi menggunakan **median dari data training** (bukan dari data testing itu sendiri)
- Scaling menggunakan **scaler yang sudah dilatih** di Tahap 1 (bukan di-fit ulang)

Kedua poin di atas penting untuk menjaga **konsistensi statistik** antara data training dan testing.

In [ ]:
# ── Simpan kolom ID untuk output prediksi ────────────────────────────────────
# Kolom ID tidak digunakan sebagai fitur, tetapi diperlukan di hasil akhir
# untuk mencocokkan prediksi dengan baris yang tepat.
test_ids = df_test[ID_COL].copy()

# ── Hapus kolom ID dari fitur ─────────────────────────────────────────────────
X_test = df_test.drop(columns=[ID_COL])

# ── Tangani missing values ───────────────────────────────────────────────────
# Gunakan median DATA TRAINING — bukan median data testing.
# Alasan: jika menggunakan median testing, statistik yang dipakai bisa
# berbeda dengan saat training, sehingga hasil scaling tidak konsisten.
missing = X_test.isnull().sum()
if missing.sum() > 0:
    print(f'⚠️  Ditemukan {missing.sum()} missing value → diisi dengan median data training.')
    X_test.fillna(train_median, inplace=True)
else:
    print('✅ Tidak ada missing values pada data testing.')

# ── Feature Scaling ──────────────────────────────────────────────────────────
# Gunakan .transform() saja — BUKAN .fit_transform()
# .fit_transform() akan menghitung ulang mean & std dari data testing,
# menghasilkan skala yang berbeda dengan training → prediksi tidak akurat.
X_test_scaled = loaded_scaler.transform(X_test)

print(f'✅ Preprocessing selesai.')
print(f'   Shape fitur testing : {X_test_scaled.shape}')

### 2.5 Prediksi

Model yang sudah dilatih digunakan untuk memprediksi kelas target (`quality`) pada data testing.

In [ ]:
# Prediksi kelas target untuk setiap baris data testing
y_pred = loaded_model.predict(X_test_scaled)

print(f'✅ Prediksi selesai! Total: {len(y_pred)} sampel')
print(f'\n📊 Distribusi hasil prediksi:')
pred_dist = pd.Series(y_pred).value_counts().sort_index()
for kelas, jumlah in pred_dist.items():
    print(f'   Quality {kelas} : {jumlah} sampel ({jumlah/len(y_pred)*100:.1f}%)')

### 2.6 Simpan & Unduh Hasil Prediksi

Hasil prediksi disimpan dalam format CSV dengan dua kolom: `Id` dan `Quality`, dipisahkan oleh titik koma (`;`) sesuai format yang diminta.

In [ ]:
# Gabungkan kolom Id dengan hasil prediksi
# Urutan baris dijamin benar karena Id dan y_pred berasal dari
# dataset yang sama tanpa ada pengacakan urutan.
hasil = pd.DataFrame({
    'Id'     : test_ids.values,
    'Quality': y_pred
})

# Simpan ke CSV dengan separator titik koma
output_file = f'hasilprediksi_{NIM_3_DIGIT}.csv'
hasil.to_csv(output_file, sep=';', index=False)

# Verifikasi — baca ulang untuk memastikan format sudah benar
verif = pd.read_csv(output_file, sep=';')
print(f'✅ File tersimpan  : {output_file}')
print(f'   Total baris    : {len(verif)}')
print(f'   Kolom          : {verif.columns.tolist()}')
print(f'   Ada nilai NaN  : {verif.isnull().sum().sum() > 0}')
print(f'\n📄 Preview 10 baris pertama:')
display(verif.head(10))

# Unduh file ke komputer secara otomatis
files.download(output_file)
print(f'\n⬇️  File {output_file} sedang diunduh...')